# 04. 구조화된 EDA 리포트 자동 생성

## 실험 목표
- `pydantic.BaseModel`로 **분석 리포트 스키마** 설계
- 에이전트 출력을 자유 텍스트 → **구조화된 JSON**으로 강제
- 리포트를 재사용 가능한 데이터 형태로 저장

## 핵심 개념: `result_type`
```python
agent = Agent(model, result_type=MySchema)
result = await agent.run(prompt)
result.output  # → MySchema 인스턴스 (dict가 아닌 객체)
```
PydanticAI는 LLM 출력을 자동으로 파싱하여 지정한 Pydantic 모델로 변환합니다.

---
## 0. 환경 준비

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import Optional
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.gemini import GeminiModel

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
print("✅ 환경 설정 완료")

---
## 1. EDA 리포트 스키마 설계

In [ ]:
# ─────────────────────────────────────────────────────────────
# EDA 리포트 Pydantic 스키마
# ─────────────────────────────────────────────────────────────

class DataOverview(BaseModel):
    """데이터 개요"""
    rows: int = Field(description="전체 행 수")
    columns: int = Field(description="전체 열 수")
    numeric_columns: list[str] = Field(description="수치형 컬럼 목록")
    categorical_columns: list[str] = Field(description="범주형 컬럼 목록")
    total_missing_cells: int = Field(description="전체 결측 셀 수")


class MissingInfo(BaseModel):
    """결측치 정보"""
    column_name: str = Field(description="컬럼명")
    missing_count: int = Field(description="결측치 개수")
    missing_ratio: float = Field(description="결측 비율 (0~100)")
    recommendation: str = Field(description="처리 권장사항 (제거/대체/유지 등)")


class OutlierInfo(BaseModel):
    """이상치 정보"""
    column_name: str = Field(description="컬럼명")
    outlier_count: int = Field(description="이상치 개수")
    outlier_ratio: float = Field(description="이상치 비율 (0~100)")
    lower_bound: float = Field(description="IQR 하한 경계")
    upper_bound: float = Field(description="IQR 상한 경계")
    recommendation: str = Field(description="처리 권장사항")


class CorrelationHighlight(BaseModel):
    """주목할 상관관계"""
    col_a: str
    col_b: str
    correlation: float
    interpretation: str = Field(description="상관관계 해석 (약/중/강, 양/음 방향)")


class KeyInsight(BaseModel):
    """주요 인사이트"""
    category: str = Field(description="인사이트 분류 (결측치/이상치/분포/상관관계 등)")
    insight: str = Field(description="발견한 인사이트 내용")
    priority: str = Field(description="중요도: 높음/중간/낮음")


class EDAReport(BaseModel):
    """전체 EDA 리포트"""
    dataset_name: str = Field(description="데이터셋 이름")
    overview: DataOverview
    missing_analysis: list[MissingInfo] = Field(description="결측치 있는 컬럼들의 분석")
    outlier_analysis: list[OutlierInfo] = Field(description="이상치 분석 결과")
    correlation_highlights: list[CorrelationHighlight] = Field(description="주목할 상관관계 (상위 3개)")
    key_insights: list[KeyInsight] = Field(description="전체 EDA에서 도출한 핵심 인사이트")
    next_steps: list[str] = Field(description="권장 후속 작업 목록")

print("✅ EDA 리포트 스키마 정의 완료")
print(f"최상위 필드: {list(EDAReport.model_fields.keys())}")

---
## 2. Tool 함수 및 에이전트 구성

In [ ]:
def describe_data(df):
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    return {"shape": {"rows": int(df.shape[0]), "columns": int(df.shape[1])},
            "numeric_columns": numeric_cols, "categorical_columns": cat_cols,
            "numeric_summary": df[numeric_cols].describe().round(2).to_dict()}

def check_missing(df):
    missing_count = df.isnull().sum()
    missing_ratio = (df.isnull().sum() / len(df) * 100).round(2)
    missing_df = pd.DataFrame({'count': missing_count, 'ratio(%)': missing_ratio})
    missing_df = missing_df.query('count > 0').sort_values('count', ascending=False)
    return {"total_missing_cells": int(df.isnull().sum().sum()),
            "columns_with_missing": missing_df.to_dict(orient='index')}

def detect_outliers_all(df):
    """모든 수치형 컬럼에 대해 일괄 이상치 탐지"""
    results = {}
    for col in df.select_dtypes(include='number').columns:
        series = df[col].dropna()
        Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
        IQR = Q3 - Q1
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        outliers = df[(df[col] < lower) | (df[col] > upper)][col]
        results[col] = {
            "outlier_count": len(outliers),
            "outlier_ratio(%)": round(len(outliers) / len(series) * 100, 2),
            "lower_bound": round(float(lower), 3),
            "upper_bound": round(float(upper), 3)
        }
    return results

def correlation_top(df, method='pearson', top_n=5):
    numeric_df = df.select_dtypes(include='number')
    corr = numeric_df.corr(method=method).round(3)
    pairs = []
    for i in range(len(corr.columns)):
        for j in range(i+1, len(corr.columns)):
            pairs.append((corr.columns[i], corr.columns[j], float(corr.iloc[i, j])))
    pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    return [{"col_a": a, "col_b": b, "corr": v} for a, b, v in pairs[:top_n]]


@dataclass
class DataDeps:
    df: pd.DataFrame
    dataset_name: str

model = GeminiModel(model_name="gemini-2.0-flash", api_key=api_key)

# result_type에 EDAReport 스키마를 지정
report_agent = Agent(
    model=model,
    deps_type=DataDeps,
    result_type=EDAReport,
    system_prompt="""
    당신은 EDA 자동화 에이전트입니다.
    제공된 Tool을 사용하여 데이터를 분석한 후,
    분석 결과를 EDAReport 스키마에 맞게 정확하고 구체적으로 채워주십시오.
    
    중요: 모든 수치는 실제 Tool 실행 결과에서 추출해야 하며,
    추측이나 가정은 사용하지 마십시오.
    """
)

@report_agent.tool
def tool_describe_data(ctx: RunContext[DataDeps]) -> dict:
    """데이터 기본 통계량을 반환합니다."""
    return describe_data(ctx.deps.df)

@report_agent.tool
def tool_check_missing(ctx: RunContext[DataDeps]) -> dict:
    """결측치 현황을 분석합니다."""
    return check_missing(ctx.deps.df)

@report_agent.tool
def tool_detect_outliers_all(ctx: RunContext[DataDeps]) -> dict:
    """모든 수치형 컬럼의 이상치를 일괄 탐지합니다."""
    return detect_outliers_all(ctx.deps.df)

@report_agent.tool
def tool_correlation_top(ctx: RunContext[DataDeps], method: str = 'pearson') -> list:
    """상위 상관관계 쌍을 반환합니다."""
    return correlation_top(ctx.deps.df, method)

print("✅ 구조화 리포트 에이전트 구성 완료")

---
## 3. 구조화 리포트 생성

In [ ]:
df = pd.read_csv("data/sample_data.csv")
deps = DataDeps(df=df, dataset_name="Titanic 승객 데이터")

async def generate_report():
    result = await report_agent.run(
        "데이터셋에 대해 완전한 EDA 리포트를 생성해주세요. "
        "모든 Tool을 활용하여 데이터를 분석하고 EDAReport 형식으로 정리하십시오.",
        deps=deps
    )
    return result.output

report = await generate_report()
print(f"✅ 리포트 생성 완료")
print(f"타입: {type(report).__name__}")

In [ ]:
# 구조화된 리포트 출력
print("=" * 60)
print(f"📋 EDA 리포트: {report.dataset_name}")
print("=" * 60)

print("\n▶ 데이터 개요")
print(f"  행: {report.overview.rows:,} | 열: {report.overview.columns}")
print(f"  수치형: {report.overview.numeric_columns}")
print(f"  범주형: {report.overview.categorical_columns}")
print(f"  총 결측 셀: {report.overview.total_missing_cells:,}")

print("\n▶ 결측치 분석")
for m in report.missing_analysis:
    print(f"  {m.column_name}: {m.missing_count}개 ({m.missing_ratio:.1f}%) → {m.recommendation}")

print("\n▶ 이상치 분석")
for o in report.outlier_analysis:
    print(f"  {o.column_name}: {o.outlier_count}개 ({o.outlier_ratio:.1f}%) [{o.lower_bound}~{o.upper_bound}]")
    print(f"    → {o.recommendation}")

print("\n▶ 주목할 상관관계")
for c in report.correlation_highlights:
    print(f"  {c.col_a} ↔ {c.col_b}: {c.correlation:.3f} ({c.interpretation})")

print("\n▶ 핵심 인사이트")
for i, insight in enumerate(report.key_insights, 1):
    print(f"  [{i}] [{insight.priority}] {insight.category}: {insight.insight}")

print("\n▶ 권장 후속 작업")
for i, step in enumerate(report.next_steps, 1):
    print(f"  {i}. {step}")

---
## 4. JSON 저장 및 재활용

In [ ]:
# 리포트를 JSON 파일로 저장
report_dict = report.model_dump()
output_path = "data/eda_report.json"

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(report_dict, f, ensure_ascii=False, indent=2)

print(f"✅ 리포트 저장 완료: {output_path}")
print()

# 저장된 JSON 일부 확인
with open(output_path, encoding='utf-8') as f:
    saved = json.load(f)

print("[저장된 JSON 구조]")
print(json.dumps({k: type(v).__name__ for k, v in saved.items()}, ensure_ascii=False, indent=2))

In [ ]:
# JSON에서 EDAReport 객체로 복원
restored_report = EDAReport.model_validate(saved)
print("✅ JSON → EDAReport 복원 성공")
print(f"데이터셋명: {restored_report.dataset_name}")
print(f"인사이트 수: {len(restored_report.key_insights)}개")

---
## 5. 정리

### 구조화 출력의 장점
| 방식 | 자유 텍스트 | 구조화 출력 |
|---|---|---|
| 재활용성 | 낮음 | **높음** (JSON 저장/로드) |
| 일관성 | 불안정 | **보장** (스키마 강제) |
| 후처리 | 파싱 필요 | **불필요** (객체로 바로 사용) |
| 자동화 | 어려움 | **용이** (파이프라인 연결) |

### 다음 노트북 (05)
실제 포트폴리오 데이터에 전체 파이프라인을 적용하고
에이전트 응답 품질과 한계점을 종합 평가한다.